# AiXbio — Notebook 6: Mechanistic Interpretability Insights

Contains deeper mechanistic insights for the paper:
- **Insight A**: SAE ↔ Probe Alignment (Which SAE features geometrically ALIGN with the probe direction?)
- **Insight B**: Cumulative DPA trajectory (How does the toxin signal build up layer by layer?)

In [ ]:
import warnings, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

main = json.load(open('results/main_results.json'))
transfer_data = main['sae']['feature_transfer']

probe_dir = np.load('results/probe_direction.npy')
tox_dpa = np.load('results/circuits/dpa_toxin.npy')
ctrl_dpa = np.load('results/circuits/dpa_control.npy')
rdsg_dpa = np.load('results/circuits/dpa_redesign.npy')

## Insight A: SAE ↔ Probe Alignment

We investigate which SAE decoder directions geometrically align with the linear probe weight.

In [ ]:
try:
    from interplm.sae.inference import load_sae_from_hf
    sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=33).to(DEVICE).eval()
    
    # SAE decoder matrix: maps features → embedding space
    # For ReLUSAE in interPLM, the decoder weight is usually (1280, 10240)
    W_dec = sae.decoder.weight.detach().cpu().numpy().T  # (10240, 1280)
    
    p_norm = probe_dir / (np.linalg.norm(probe_dir) + 1e-8)
    alignments = (W_dec * p_norm).sum(-1)     # (10240,) cosine with probe
    
    top_aligned  = np.argsort(alignments)[::-1][:10]
    top_opposing = np.argsort(alignments)[:10]
    
    print('Top SAE features ALIGNED with probe direction (push toward toxic):')
    for f in top_aligned:
        xfer = next((d['transfer_ratio'] for d in transfer_data if d['feature']==f), 'N/A')
        print(f'  Feature {f:5d}: alignment={alignments[f]:+.3f}  transfer={xfer}')
    
    print('\nTop SAE features OPPOSING probe direction (push toward safe):')
    for f in top_opposing:
        xfer = next((d['transfer_ratio'] for d in transfer_data if d['feature']==f), 'N/A')
        print(f'  Feature {f:5d}: alignment={alignments[f]:+.3f}  transfer={xfer}')
        
except ImportError:
    print("interplm not installed. Skipping SAE alignment analysis.")
except AttributeError as e:
    print(f"Attribute error when accessing SAE weights: {e}")

## Insight B: Cumulative DPA Trajectory

Layer-by-layer cumulative probe score accumulation.

In [ ]:
tox_traj  = np.cumsum(tox_dpa.mean(0))   # (33,) — running total for toxins
ctrl_traj = np.cumsum(ctrl_dpa.mean(0))
rdsg_traj = np.cumsum(rdsg_dpa.mean(0))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1,34), tox_traj,  color='#E07B39', lw=2.2, label='Natural toxins')
ax.plot(range(1,34), rdsg_traj, color='#C0392B', lw=2.2, label='Redesigns',  ls='--')
ax.plot(range(1,34), ctrl_traj, color='#2E86AB', lw=2.2, label='Controls')
ax.axhline(0, color='black', lw=0.8)

for l in [17,19,20,30,31,32]:
    ax.axvspan(l-0.5, l+0.5, alpha=0.1, color='#E07B39')

ax.set_xlabel('ESM-2 Layer')
ax.set_ylabel('Cumulative DPA')
ax.set_title('Figure 6 — Toxin signal accumulation through ESM-2 depth\n(shaded = primary discrimination layers)')
ax.legend()

Path('figures').mkdir(exist_ok=True)
plt.savefig('figures/fig6_dpa_trajectory.pdf', bbox_inches='tight')
plt.savefig('figures/fig6_dpa_trajectory.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved figures/fig6_dpa_trajectory.pdf and .png")